# 🧠 L6 — Donner une mémoire courte à un agent

> ⏱️ **Durée indicative : 30 à 45 minutes**  
> 🎯 **Objectif :** comprendre ce que LangChain mémorise, ce que Mistral reçoit et pourquoi un `thread_id` sépare deux conversations.

Nous allons comparer trois situations : un agent **sans mémoire**, le même agent dans **le même fil**, puis dans **un nouveau fil**.

Le scénario Chinook du cours reste notre fil rouge : **Frank Harris** demande sa dernière facture, puis « Quels étaient les titres ? » 🎵

## 🧠 ELI5 — La mémoire est un dossier, pas un nouveau cerveau

Imaginez un conseiller avec plusieurs chemises cartonnées :

- chaque `thread_id` est l'étiquette d'une chemise ;
- le **checkpointer** range les messages dans la bonne chemise ;
- avant la réponse suivante, LangGraph ressort cette chemise et redonne son contenu au modèle.

⚠️ **Le modèle n'apprend pas et ses poids ne changent pas.** L'application persiste un état, puis le renvoie au modèle comme contexte. La [mémoire court terme officielle de LangChain](https://docs.langchain.com/oss/python/langchain/short-term-memory) décrit ce mécanisme par fil.

### 🗺️ Schéma mental

```text
question 1 ─┐                         ┌─> Mistral répond
réponse 1  ─┼─> thread_id "frank" ─> checkpointer ─> historique remis au modèle
question 2 ─┘

question 2 ───> nouveau thread_id ─> dossier vide ─> aucun contexte sur Frank
```

## 🛠️ 0. Préparer Mistral sans lire de fichier `.env`

Le notebook utilise uniquement `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL`, déjà présentes dans le processus. Il ne lit, n'affiche et ne modifie aucun fichier `.env`.

Nous utilisons [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai), l'intégration officielle LangChain pour Mistral, avec `mistral-medium-latest` et `temperature=0`.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

from langchain_mistralai import ChatMistralAI

# 📚 https://docs.langchain.com/oss/python/integrations/chat/mistralai
MODEL = "mistral-medium-latest"

required_variables = ("MISTRAL_API_KEY", "MISTRAL_SERVER_URL")
missing_variables = [name for name in required_variables if not os.environ.get(name)]
if missing_variables:
    raise RuntimeError(
        "Variables manquantes dans le processus : " + ", ".join(missing_variables)
    )


def normalize_mistral_endpoint(raw_url: str) -> str:
    """Ajoute /v1 une seule fois au serveur compatible avec l’API Mistral."""
    base_url = raw_url.strip().rstrip("/")
    return base_url if base_url.endswith("/v1") else f"{base_url}/v1"


mistral_model = ChatMistralAI(
    model=MODEL,
    temperature=0,
    api_key=os.environ["MISTRAL_API_KEY"],
    endpoint=normalize_mistral_endpoint(os.environ["MISTRAL_SERVER_URL"]),
)
print(f"✅ Modèle configuré : {MODEL} (temperature=0)")

✅ Modèle configuré : mistral-medium-latest (temperature=0)


### 👀 Résultat attendu

La cellule confirme seulement le nom du modèle et sa température. Aucune clé ni URL sensible ne doit apparaître.

## 🛠️ 1. Ouvrir Chinook en lecture seule

Chinook joue le rôle d'un classeur de factures. Nous l'ouvrons en **lecture seule** : même si une mauvaise requête était proposée, SQLite refuserait toute modification. C'est une ceinture de sécurité technique, plus fiable qu'une phrase dans un prompt. 🔒

In [2]:
from pathlib import Path

from util.sql_db import SQLDatabase

db_path = Path("data/Chinook.db").resolve()
if not db_path.is_file():
    raise FileNotFoundError(f"Base Chinook introuvable : {db_path}")

# `mode=ro` impose la lecture seule au niveau SQLite.
readonly_uri = f"sqlite:///file:{db_path.as_posix()}?mode=ro&uri=true"
db = SQLDatabase.from_uri(readonly_uri)
print("✅ Base ouverte en lecture seule")
print("Tables :", ", ".join(db.get_usable_table_names()))

✅ Base ouverte en lecture seule
Tables : Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


### 🔍 Donner une carte de la base à l'agent

Le modèle ne voit pas spontanément les tables. Sans carte, il peut inventer `invoices` alors que la vraie table s'appelle `Invoice`. Nous découvrons explicitement les quatre tables utiles et transmettons leur schéma dans le prompt.

In [3]:
USEFUL_TABLES = ["Customer", "Invoice", "InvoiceLine", "Track"]
schema_context = db.get_table_info(USEFUL_TABLES)

print("✅ Schéma découvert pour :", ", ".join(USEFUL_TABLES))
print("- Customer.CustomerId → Invoice.CustomerId")
print("- Invoice.InvoiceId → InvoiceLine.InvoiceId")
print("- InvoiceLine.TrackId → Track.TrackId")

✅ Schéma découvert pour : Customer, Invoice, InvoiceLine, Track
- Customer.CustomerId → Invoice.CustomerId
- Invoice.InvoiceId → InvoiceLine.InvoiceId
- InvoiceLine.TrackId → Track.TrackId


### 🤔 Pause prédiction

Quel composant calcule réellement le total : Mistral ou SQLite ? Pourquoi commencer par une requête de référence sans agent ?

Réponse : SQLite lit les données. La requête de référence constitue une **boussole déterministe** pour contrôler ensuite la réponse variable du modèle.

In [4]:
REFERENCE_LAST_INVOICE = """
SELECT i.InvoiceId, i.InvoiceDate, i.Total
FROM Invoice AS i
JOIN Customer AS c ON c.CustomerId = i.CustomerId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
ORDER BY i.InvoiceDate DESC, i.InvoiceId DESC
LIMIT 1
"""

REFERENCE_TITLES = """
SELECT t.Name
FROM Track AS t
JOIN InvoiceLine AS il ON il.TrackId = t.TrackId
WHERE il.InvoiceId = 374
ORDER BY il.InvoiceLineId
"""

print("Dernière facture :", db.run(REFERENCE_LAST_INVOICE))
print("Titres :", db.run(REFERENCE_TITLES))

Dernière facture : [(374, '2013-07-04 00:00:00', 5.94)]
Titres : [('Holier Than Thou',), ('Through The Never',), ('My Friend Of Misery',), ('The Wait',), ('Blitzkrieg',), ('So What',)]


### 👀 Résultat attendu

- facture `374`, datée du `2013-07-04`, total `5.94` ;
- six titres, de `Holier Than Thou` à `So What`.

⚠️ Ce sont nos **oracles de test**. La formulation de Mistral pourra varier, mais pas ces données.

## 🛠️ 2. Construire un outil SQL sûr

Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) transforme une fonction Python en outil décrit au modèle. Le [`runtime context`](https://docs.langchain.com/oss/python/langchain/runtime) injecte la connexion sans la placer dans les arguments choisis par Mistral.

In [5]:
from dataclasses import dataclass

from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@dataclass
class RuntimeContext:
    db: SQLDatabase


FORBIDDEN_SQL = {
    "insert", "update", "delete", "alter", "drop", "create",
    "replace", "truncate", "attach", "detach", "pragma",
}


def validate_readonly_query(query: str) -> str:
    """Valide une unique requête SELECT/CTE et renvoie sa forme nettoyée."""
    statement = query.strip()
    body = statement[:-1].strip() if statement.endswith(";") else statement
    lowered = body.lower()
    if not lowered.startswith(("select ", "with ")):
        raise ValueError("Seules les requêtes SELECT ou WITH sont autorisées.")
    if ";" in body:
        raise ValueError("Une seule instruction SQL est autorisée.")
    words = set(lowered.replace("(", " ").replace(")", " ").split())
    if words.intersection(FORBIDDEN_SQL):
        raise ValueError("La requête contient une opération interdite.")
    return body


# 📚 Tools : https://docs.langchain.com/oss/python/langchain/tools
# 📚 Runtime : https://docs.langchain.com/oss/python/langchain/runtime
@tool
def execute_sql(query: str) -> str:
    """Exécute une unique requête SQLite en lecture seule sur Chinook."""
    runtime = get_runtime(RuntimeContext)
    safe_query = validate_readonly_query(query)
    try:
        return runtime.context.db.run(safe_query)
    except Exception as error:
        return f"Erreur SQL : {error}"

### 🔍 Qui fait quoi ?

| Acteur | Responsabilité |
|---|---|
| **Mistral** | choisit `execute_sql` et propose les arguments SQL |
| **LangChain/LangGraph** | transporte les messages, orchestre la boucle et injecte le contexte |
| **Python + SQLite** | valident puis exécutent réellement la requête |

Le guide officiel Mistral décrit ces étapes de [function calling](https://docs.mistral.ai/studio/conversations/function-calling).

## 🛠️ 3. Donner le schéma exact dans le prompt

Le schéma agit comme le plan d'un bâtiment : il réduit les noms de tables inventés et aide le modèle à produire les bonnes jointures.

In [6]:
SYSTEM_PROMPT = f"""Tu es un analyste SQLite prudent.

Règles :
- Utilise `execute_sql` dès qu’une réponse dépend de Chinook.
- Produis une seule requête SELECT ou WITH, jamais de modification.
- Utilise exactement les noms singuliers et la casse du schéma.
- Limite à 10 lignes, sauf demande explicite contraire.
- Si l’outil renvoie `Erreur SQL`, corrige la requête une fois.
- Ne devine aucune donnée absente du résultat.
- Réponds en français et explique brièvement le résultat.

Relations :
- Customer.CustomerId = Invoice.CustomerId
- Invoice.InvoiceId = InvoiceLine.InvoiceId
- InvoiceLine.TrackId = Track.TrackId

Schéma disponible :
{schema_context}
"""

## ▶️ 4. Expérience A — Sans mémoire

[`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) construit la boucle modèle → outil → modèle. Nous ne lui donnons encore aucun checkpointer.

In [7]:
from langchain.agents import create_agent

# 📚 https://docs.langchain.com/oss/python/langchain/agents
agent_sans_memoire = create_agent(
    model=mistral_model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

QUESTION_FRANK = "Je suis Frank Harris. Quel était le total de ma dernière facture ?"
QUESTION_TITRES = "Quels étaient les titres de cette facture ?"

### 🤔 Pause prédiction

Les deux appels suivants sont indépendants. Au second appel, que désigne « cette facture » ? Notez votre hypothèse avant d'exécuter.

In [8]:
resultat_sans_memoire_1 = agent_sans_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_FRANK}]},
    context=RuntimeContext(db=db),
)
print("1️⃣", resultat_sans_memoire_1["messages"][-1].content)

resultat_sans_memoire_2 = agent_sans_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_TITRES}]},
    context=RuntimeContext(db=db),
)
print("2️⃣", resultat_sans_memoire_2["messages"][-1].content)

1️⃣ Votre dernière facture, Frank Harris, avait un total de **5,94 €**. Cela correspond à la facture la plus récente associée à votre identifiant client.


2️⃣ Les titres de la facture avec l'**InvoiceId = 1** étaient :
1. **Balls to the Wall**
2. **Restless and Wild**

Ces titres proviennent des pistes associées à cette facture via la table `InvoiceLine`.


In [9]:
def human_messages(result: dict) -> list[str]:
    """Extrait les textes utilisateur pour observer l’état objectivement."""
    return [
        message.content
        for message in result["messages"]
        if getattr(message, "type", None) == "human"
    ]


messages_sans_memoire = human_messages(resultat_sans_memoire_2)
assert QUESTION_FRANK not in messages_sans_memoire
print("🔍 Messages utilisateur visibles au second appel :", messages_sans_memoire)

🔍 Messages utilisateur visibles au second appel : ['Quels étaient les titres de cette facture ?']


### 👀 Observation

Mistral peut demander une précision ou expliquer qu'il lui manque le client. La preuve stable est la liste affichée : **la question sur Frank n'est pas dans l'état du second appel**.

⚠️ Ne testez pas la mémoire uniquement avec une phrase générée : inspectez l'état.

## ▶️ 5. Expérience B — Même `thread_id`, avec mémoire

Nous ajoutons [`InMemorySaver`](https://docs.langchain.com/oss/python/langchain/short-term-memory). Il conserve l'état en mémoire vive pendant l'exécution du kernel. En production, on choisirait un stockage persistant adapté.

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

# 📚 https://docs.langchain.com/oss/python/langchain/short-term-memory
checkpointer = InMemorySaver()
agent_avec_memoire = create_agent(
    model=mistral_model,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=checkpointer,
)
CONFIG_FRANK = {"configurable": {"thread_id": "frank-demo-01"}}

### 🤔 Pause prédiction

Les deux appels utiliseront `frank-demo-01`. Combien de messages utilisateur seront présents après le second appel ? « Cette facture » pourra-t-elle être reliée à Frank Harris ?

In [11]:
resultat_memoire_1 = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_FRANK}]},
    config=CONFIG_FRANK,
    context=RuntimeContext(db=db),
)
print("1️⃣", resultat_memoire_1["messages"][-1].content)

resultat_memoire_2 = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_TITRES}]},
    config=CONFIG_FRANK,
    context=RuntimeContext(db=db),
)
print("2️⃣", resultat_memoire_2["messages"][-1].content)

1️⃣ Votre dernière facture, Frank Harris, avait un total de **5,94 €**.


2️⃣ Les titres de votre dernière facture, Frank Harris, étaient :
1. **Holier Than Thou**
2. **Through The Never**
3. **My Friend Of Misery**


In [12]:
messages_meme_fil = human_messages(resultat_memoire_2)
assert QUESTION_FRANK in messages_meme_fil
assert QUESTION_TITRES in messages_meme_fil
print("✅ Les deux questions sont dans le même état :")
for index, message in enumerate(messages_meme_fil, start=1):
    print(f"  {index}. {message}")

✅ Les deux questions sont dans le même état :
  1. Je suis Frank Harris. Quel était le total de ma dernière facture ?
  2. Quels étaient les titres de cette facture ?


### 👀 Observation

LangGraph a rechargé l'historique avant l'appel du modèle. Mistral peut relier « cette facture » à Frank et retrouver les six titres.

✅ C'est de la **persistance de contexte**, pas un entraînement du modèle.

## ▶️ 6. Expérience C — Nouveau `thread_id`, dossier vide

Nous gardons le même agent et le même checkpointer, mais changeons l'étiquette de la chemise.

### 🤔 Pause prédiction

Si le nouveau fil reçoit directement « Quels étaient les titres de cette facture ? », doit-il voir la question sur Frank ? Pourquoi ?

In [13]:
CONFIG_NOUVEAU_FIL = {
    "configurable": {"thread_id": "conversation-independante-01"}
}
resultat_nouveau_fil = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_TITRES}]},
    config=CONFIG_NOUVEAU_FIL,
    context=RuntimeContext(db=db),
)
messages_nouveau_fil = human_messages(resultat_nouveau_fil)
assert QUESTION_FRANK not in messages_nouveau_fil
assert messages_nouveau_fil == [QUESTION_TITRES]
print("Réponse :", resultat_nouveau_fil["messages"][-1].content)
print("🔍 État du nouveau fil :", messages_nouveau_fil)

Réponse : Les titres de la facture **n°1** étaient :
- **Balls to the Wall**
- **Restless and Wild**

*(Résultat obtenu en joignant les tables `InvoiceLine` et `Track` pour l'`InvoiceId = 1`.)*
🔍 État du nouveau fil : ['Quels étaient les titres de cette facture ?']


## 🔍 Comparaison des trois expériences

| Expérience | Checkpointer | `thread_id` partagé | Question sur Frank visible au tour 2 ? |
|---|---:|---:|---:|
| A — sans mémoire | ❌ | sans objet | ❌ |
| B — même fil | ✅ | ✅ | ✅ |
| C — nouveau fil | ✅ | ❌ | ❌ |

🧠 **Formule à retenir :** `checkpointer` + même `thread_id` = historique récupérable dans ce fil.

## 🧪 Micro-exercice — Le dossier de Julia Barnett

1. Dans un nouveau fil, demandez le total de sa dernière facture.
2. Dans le **même fil**, demandez le titre de cette facture.
3. Inspectez les messages utilisateur pour prouver que les deux questions sont mémorisées.

💡 Utilisez `agent_avec_memoire`, un `thread_id` inédit et `RuntimeContext(db=db)`.

In [14]:
QUESTION_JULIA_1 = "Je suis Julia Barnett. Quel était le total de ma dernière facture ?"
QUESTION_JULIA_2 = "Quel était le titre acheté sur cette facture ?"
CONFIG_JULIA = {"configurable": {"thread_id": "TODO-julia-fil-unique"}}

# TODO 1 : invoquez l’agent avec QUESTION_JULIA_1 et CONFIG_JULIA.
# TODO 2 : invoquez-le avec QUESTION_JULIA_2 et le MÊME CONFIG_JULIA.
# TODO 3 : affichez human_messages(...) pour prouver la mémoire.

### ✅ Critères de réussite

- les deux appels réutilisent exactement le même `thread_id` ;
- l'état final contient les deux questions ;
- la dernière facture est la `363`, d'un total de `0.99` ;
- le titre est `You've Got Another Thing Comin'`.

<details>
<summary>✅ Afficher la correction</summary>

```python
resultat_julia_1 = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_JULIA_1}]},
    config=CONFIG_JULIA,
    context=RuntimeContext(db=db),
)
resultat_julia_2 = agent_avec_memoire.invoke(
    {"messages": [{"role": "user", "content": QUESTION_JULIA_2}]},
    config=CONFIG_JULIA,
    context=RuntimeContext(db=db),
)
messages_julia = human_messages(resultat_julia_2)
assert QUESTION_JULIA_1 in messages_julia
assert QUESTION_JULIA_2 in messages_julia
print(resultat_julia_2["messages"][-1].content)
print(messages_julia)
```

</details>

## ✅ Acquis de la leçon

- Sans checkpointer, un nouvel appel ne récupère pas automatiquement le précédent.
- `InMemorySaver` conserve l'état tant que le processus vit.
- Le `thread_id` choisit la conversation à reprendre.
- Changer de fil isole les conversations.
- La mémoire n'entraîne pas Mistral : elle lui redonne des messages.
- Lecture seule et schéma explicite fiabilisent l'exemple SQL.

## 🧭 Transition vers L7

Nous savons conserver le contexte. Dans L7, nous demanderons une réponse **structurée et validable**, plutôt qu'un simple texte libre.

## 📚 Documentation officielle

- [LangChain — Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)
- [LangChain — Agents et `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Tools et `@tool`](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Runtime context](https://docs.langchain.com/oss/python/langchain/runtime)
- [LangChain — `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral — Function calling](https://docs.mistral.ai/studio/conversations/function-calling)